In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

import joblib

import warnings
warnings.filterwarnings("ignore")

In [4]:
TRAIN_PATH = "../datasets/FI2010/train_clean.csv"
TEST_PATH = "../datasets/FI2010/test_clean.csv"

PROCESSED_DIR = "../processed/FI2010"
ARTIFACT_DIR = "../artifacts"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)


In [5]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

Train Shape: (362400, 149)
Test Shape : (31937, 149)


In [6]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

Train Shape: (362400, 149)
Test Shape : (31937, 149)


In [7]:
feature_columns = [str(i) for i in range(143)]
target_columns = [str(i) for i in range(144, 149)]

print("Number of input features:", len(feature_columns))
print("Target columns:", target_columns)

Number of input features: 143
Target columns: ['144', '145', '146', '147', '148']


In [8]:
print("Unique values in column 143:")
print(train_df["143"].unique())

print("\nNumber of unique values:")
print(train_df["143"].nunique())

Unique values in column 143:
[0.]

Number of unique values:
1


In [9]:
train_df = train_df.drop(columns=["143"])
test_df = test_df.drop(columns=["143"])

print("Train Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

Train Shape: (362400, 148)
Test Shape : (31937, 148)


In [10]:
TARGET_COLUMN = "148"

X_train = train_df[feature_columns].copy()
y_train = train_df[TARGET_COLUMN].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[TARGET_COLUMN].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (362400, 143)
y_train: (362400,)
X_test : (31937, 143)
y_test : (31937,)


In [11]:
label_mapping = {
    1: 0,
    2: 1,
    3: 2
}

y_train = y_train.map(label_mapping)
y_test = y_test.map(label_mapping)

print("Training labels after encoding:")
print(sorted(y_train.unique()))

print("\nTesting labels after encoding:")
print(sorted(y_test.unique()))

Training labels after encoding:
[np.int64(0), np.int64(1), np.int64(2)]

Testing labels after encoding:
[np.int64(0), np.int64(1), np.int64(2)]


In [12]:
validation_ratio = 0.20

split_index = int(len(X_train) * (1 - validation_ratio))

X_train_raw = X_train.iloc[:split_index].copy()
y_train_raw = y_train.iloc[:split_index].copy()

X_val_raw = X_train.iloc[split_index:].copy()
y_val_raw = y_train.iloc[split_index:].copy()

print("Training:", X_train_raw.shape)
print("Validation:", X_val_raw.shape)
print("Testing:", X_test.shape)

Training: (289920, 143)
Validation: (72480, 143)
Testing: (31937, 143)


In [13]:
print("Training starts at row:", X_train_raw.index[0])
print("Training ends at row:", X_train_raw.index[-1])

print("\nValidation starts at row:", X_val_raw.index[0])
print("Validation ends at row:", X_val_raw.index[-1])

print("\nTesting starts at row:", X_test.index[0])
print("Testing ends at row:", X_test.index[-1])

Training starts at row: 0
Training ends at row: 289919

Validation starts at row: 289920
Validation ends at row: 362399

Testing starts at row: 0
Testing ends at row: 31936


In [14]:
print("Training labels:")
print(sorted(y_train.unique()))

print("\nTesting labels:")
print(sorted(y_test.unique()))

Training labels:
[np.int64(0), np.int64(1), np.int64(2)]

Testing labels:
[np.int64(0), np.int64(1), np.int64(2)]


#### Pipeline Created

In [15]:
numeric_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, feature_columns)
    ],
    remainder="drop"
)

In [17]:
X_train_scaled = preprocessor.fit_transform(X_train_raw)

X_val_scaled = preprocessor.transform(X_val_raw)

X_test_scaled = preprocessor.transform(X_test)

print("Scaled Train Shape:", X_train_scaled.shape)
print("Scaled Validation Shape:", X_val_scaled.shape)
print("Scaled Test Shape:", X_test_scaled.shape)

Scaled Train Shape: (289920, 143)
Scaled Validation Shape: (72480, 143)
Scaled Test Shape: (31937, 143)


In [18]:
X_train_scaled = np.asarray(X_train_scaled, dtype=np.float32)
X_val_scaled = np.asarray(X_val_scaled, dtype=np.float32)
X_test_scaled = np.asarray(X_test_scaled, dtype=np.float32)

y_train_raw = y_train_raw.to_numpy(dtype=np.int64)
y_val_raw = y_val_raw.to_numpy(dtype=np.int64)
y_test = y_test.to_numpy(dtype=np.int64)

In [19]:
SEQUENCE_LENGTH = 100

In [20]:
def create_sequences(X, y, sequence_length):

    X_sequences = []
    y_sequences = []

    for i in range(sequence_length - 1, len(X)):
        X_sequences.append(
            X[i - sequence_length + 1:i + 1]
        )

        y_sequences.append(
            y[i]
        )

    return (
        np.array(X_sequences, dtype=np.float32),
        np.array(y_sequences, dtype=np.int64)
    )

Convert time-series data into sliding windows.

    X shape:
        (samples, features)

    y shape:
        (samples,)

    Output X:
        (samples, sequence_length, features)

    Output y:
        (samples,)

In [21]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_raw,
    SEQUENCE_LENGTH
)

print("X_train_seq:", X_train_seq.shape)
print("y_train_seq:", y_train_seq.shape)

X_train_seq: (289821, 100, 143)
y_train_seq: (289821,)


In [22]:
X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_raw,
    SEQUENCE_LENGTH
)

print("X_val_seq:", X_val_seq.shape)
print("y_val_seq:", y_val_seq.shape)

X_val_seq: (72381, 100, 143)
y_val_seq: (72381,)


In [23]:
X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test,
    SEQUENCE_LENGTH
)

print("X_test_seq:", X_test_seq.shape)
print("y_test_seq:", y_test_seq.shape)

X_test_seq: (31838, 100, 143)
y_test_seq: (31838,)


In [24]:
print("FINAL FI-2010 SHAPES")

print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val  :", X_val_seq.shape)
print("y_val  :", y_val_seq.shape)

print("X_test :", X_test_seq.shape)
print("y_test :", y_test_seq.shape)

FINAL FI-2010 SHAPES
X_train: (289821, 100, 143)
y_train: (289821,)
X_val  : (72381, 100, 143)
y_val  : (72381,)
X_test : (31838, 100, 143)
y_test : (31838,)


In [25]:
print("Train labels:", np.unique(y_train_seq))
print("Validation labels:", np.unique(y_val_seq))
print("Test labels:", np.unique(y_test_seq))

Train labels: [0 1 2]
Validation labels: [0 1 2]
Test labels: [0 1 2]


In [26]:
print("Training distribution:")
print(pd.Series(y_train_seq).value_counts(normalize=True).sort_index())

print("\nValidation distribution:")
print(pd.Series(y_val_seq).value_counts(normalize=True).sort_index())

print("\nTesting distribution:")
print(pd.Series(y_test_seq).value_counts(normalize=True).sort_index())

Training distribution:
0    0.397193
1    0.215292
2    0.387515
Name: proportion, dtype: float64

Validation distribution:
0    0.331593
1    0.363396
2    0.305011
Name: proportion, dtype: float64

Testing distribution:
0    0.383347
1    0.274295
2    0.342358
Name: proportion, dtype: float64


In [27]:
np.save(
    os.path.join(PROCESSED_DIR, "X_train.npy"),
    X_train_seq
)

np.save(
    os.path.join(PROCESSED_DIR, "y_train.npy"),
    y_train_seq
)

np.save(
    os.path.join(PROCESSED_DIR, "X_val.npy"),
    X_val_seq
)

np.save(
    os.path.join(PROCESSED_DIR, "y_val.npy"),
    y_val_seq
)

np.save(
    os.path.join(PROCESSED_DIR, "X_test.npy"),
    X_test_seq
)

np.save(
    os.path.join(PROCESSED_DIR, "y_test.npy"),
    y_test_seq
)

print("FI-2010 processed data saved successfully!")

FI-2010 processed data saved successfully!


In [28]:
joblib.dump(
    preprocessor,
    os.path.join(ARTIFACT_DIR, "fi2010_preprocessor.pkl")
)

print("Preprocessor saved successfully!")

Preprocessor saved successfully!


#### PART B — Financial PhraseBank

Now we prepare the text dataset for FinBERT.

In [29]:
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

In [30]:
PHRASEBANK_PATH = (
    "../datasets/FinancialPhraseBank-v1.0/"
    "Sentences_AllAgree.txt"
)

phrase_df = pd.read_csv(
    PHRASEBANK_PATH,
    sep="@",
    header=None,
    names=["Sentence", "Sentiment"],
    encoding="latin-1",
    engine="python"
)

print("Shape:", phrase_df.shape)

display(phrase_df.head())

Shape: (2264, 2)


,Sentence,Sentiment
0,"According to Gran , the company has no plans t...",neutral
1,"For the last quarter of 2010 , Componenta 's n...",positive
2,"In the third quarter of 2010 , net sales incre...",positive
3,Operating profit rose to EUR 13.1 mn from EUR ...,positive
4,"Operating profit totalled EUR 21.1 mn , up fro...",positive


In [31]:
phrase_df["Sentence"] = phrase_df["Sentence"].astype(str).str.strip()
phrase_df["Sentiment"] = phrase_df["Sentiment"].astype(str).str.strip()

phrase_df = phrase_df.dropna(
    subset=["Sentence", "Sentiment"]
)

phrase_df = phrase_df.drop_duplicates(
    subset=["Sentence", "Sentiment"]
).reset_index(drop=True)

print("Final Shape:", phrase_df.shape)

Final Shape: (2259, 2)


In [32]:
sentiment_mapping = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

phrase_df["label"] = phrase_df["Sentiment"].map(
    sentiment_mapping
)

print(phrase_df["label"].value_counts().sort_index())

label
0     303
1    1386
2     570
Name: count, dtype: int64


In [33]:
display(
    phrase_df[
        ["Sentence", "Sentiment", "label"]
    ].head(10)
)

,Sentence,Sentiment,label
0,"According to Gran , the company has no plans t...",neutral,1
1,"For the last quarter of 2010 , Componenta 's n...",positive,2
2,"In the third quarter of 2010 , net sales incre...",positive,2
3,Operating profit rose to EUR 13.1 mn from EUR ...,positive,2
4,"Operating profit totalled EUR 21.1 mn , up fro...",positive,2
5,Finnish Talentum reports its operating profit ...,positive,2
6,Clothing retail chain Sepp+ñl+ñ 's sales incre...,positive,2
7,Consolidated net sales increased 16 % to reach...,positive,2
8,Foundries division reports its sales increased...,positive,2
9,"HELSINKI ( AFX ) - Shares closed higher , led ...",positive,2


In [34]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    phrase_df["Sentence"].tolist(),
    phrase_df["label"].tolist(),
    test_size=0.20,
    random_state=42,
    stratify=phrase_df["label"]
)

print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))

Training samples: 1807
Validation samples: 452


In [35]:
MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("FinBERT tokenizer loaded successfully!")

FinBERT tokenizer loaded successfully!


### Choose Maximum Sequence Length

From our EDA, we analyzed sentence lengths.

For the first implementation, we'll use:

This means sentences longer than 128 tokens will be truncated.

Shorter sentences will be padded.

In [36]:
MAX_LENGTH = 128

In [37]:
train_encodings = tokenizer(
    train_texts,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

print("input_ids shape:")
print(train_encodings["input_ids"].shape)

print("\nattention_mask shape:")
print(train_encodings["attention_mask"].shape)

input_ids shape:
torch.Size([1807, 128])

attention_mask shape:
torch.Size([1807, 128])


In [38]:
val_encodings = tokenizer(
    val_texts,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

print("input_ids shape:")
print(val_encodings["input_ids"].shape)

print("\nattention_mask shape:")
print(val_encodings["attention_mask"].shape)

input_ids shape:
torch.Size([452, 128])

attention_mask shape:
torch.Size([452, 128])


In [39]:
import torch
from torch.utils.data import Dataset


class FinancialPhraseBankDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(
            labels,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: value[idx]
            for key, value in self.encodings.items()
        }

        item["labels"] = self.labels[idx]

        return item

In [40]:
train_dataset = FinancialPhraseBankDataset(
    train_encodings,
    train_labels
)

val_dataset = FinancialPhraseBankDataset(
    val_encodings,
    val_labels
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Training dataset: 1807
Validation dataset: 452


In [41]:
sample = train_dataset[0]

print("Keys:")
print(sample.keys())

print("\nInput IDs shape:")
print(sample["input_ids"].shape)

print("\nAttention mask shape:")
print(sample["attention_mask"].shape)

print("\nLabel:")
print(sample["labels"])

Keys:
dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])

Input IDs shape:
torch.Size([128])

Attention mask shape:
torch.Size([128])

Label:
tensor(2)


#### PART C — Save PhraseBank Preprocessing

In [42]:
PHRASE_PROCESSED_DIR = "../processed/PhraseBank"

os.makedirs(
    PHRASE_PROCESSED_DIR,
    exist_ok=True
)

torch.save(
    {
        "input_ids": train_encodings["input_ids"],
        "attention_mask": train_encodings["attention_mask"],
        "labels": torch.tensor(
            train_labels,
            dtype=torch.long
        )
    },
    os.path.join(
        PHRASE_PROCESSED_DIR,
        "train.pt"
    )
)

torch.save(
    {
        "input_ids": val_encodings["input_ids"],
        "attention_mask": val_encodings["attention_mask"],
        "labels": torch.tensor(
            val_labels,
            dtype=torch.long
        )
    },
    os.path.join(
        PHRASE_PROCESSED_DIR,
        "val.pt"
    )
)

print("PhraseBank processed data saved successfully!")

PhraseBank processed data saved successfully!


##### PART D — Final Verification

In [43]:
print("PREPROCESSING SUMMARY")
print("\nFI-2010")
print("Train:", X_train_seq.shape)
print("Validation:", X_val_seq.shape)
print("Test:", X_test_seq.shape)
print("Features:", X_train_seq.shape[-1])
print("Sequence Length:", X_train_seq.shape[1])
print("Classes:", np.unique(y_train_seq))

print("\nFinancial PhraseBank")
print("Train:", train_encodings["input_ids"].shape)
print("Validation:", val_encodings["input_ids"].shape)
print("Maximum Length:", MAX_LENGTH)
print("Classes:", sorted(set(train_labels)))
print("Preprocessing completed successfully!")

PREPROCESSING SUMMARY

FI-2010
Train: (289821, 100, 143)
Validation: (72381, 100, 143)
Test: (31838, 100, 143)
Features: 143
Sequence Length: 100
Classes: [0 1 2]

Financial PhraseBank
Train: torch.Size([1807, 128])
Validation: torch.Size([452, 128])
Maximum Length: 128
Classes: [0, 1, 2]
Preprocessing completed successfully!
